In [ ]:
import os
import glob
import pandas as pd
from fitparse import FitFile
from tqdm import tqdm


In [ ]:
def convert_fit_dir_to_csv(dir_path: str, output_dir: str = "./Dataframes") -> None:
    """
    Read all .fit files in a directory and convert them into two CSV files.

    This function searches `dir_path` for `.fit` files and parses their
    messages using `fitparse`. Two types of data are distinguished:

    - "record" messages: second-by-second workout/training data
      (e.g. heart rate, speed, position at each timestamp).
    - all other messages (e.g. "session", "lap", "event"):
      higher-level information about the overall ride/activity.

    Both datasets are collected separately and saved as CSV files
    in the specified output directory:
    - workouts_per_second.csv (record messages)
    - ride_records.csv (other messages)

    If an error occurs while reading a file (e.g. a corrupted file),
    that file is skipped and an error message is printed, without
    aborting the entire process.

    Parameters
    ----------
    dir_path : str
        Path to the directory containing the .fit files.
    output_dir : str, optional
        Path to the directory where the resulting CSV files should be
        saved. Created automatically if it does not exist yet.
        Default: "./Dataframes".

    Returns
    -------
    None
        The function does not return any values. Results are saved
        exclusively as CSV files.

    Examples
    --------
    >>> fit_dir_to_dataframe("./Source_Data/Fitnessdaten")
    Processing FIT files: 100%|██████████| 42/42 [00:15<00:00,  2.75it/s]
    """
    os.makedirs(output_dir, exist_ok=True)

    all_fit_records = []
    ride_records = []

    fit_files = glob.glob(os.path.join(dir_path, "*.fit"))

    for file_path in tqdm(fit_files, desc="Processing FIT files"):
        try:
            fitfile = FitFile(file_path)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        file_name = os.path.basename(file_path)

        for message in fitfile.get_messages():
            row = {"source_file": file_name, "message_type": message.name}
            for field in message:
                row[field.name] = field.value

            if message.name == "record":
                all_fit_records.append(row)
            elif message.name == "session":
                ride_records.append(row)

    pd.DataFrame(all_fit_records).to_csv(
        os.path.join(output_dir, "workouts_per_second.csv"), index=False
    )
    pd.DataFrame(ride_records).to_csv(
        os.path.join(output_dir, "session_records.csv"), index=False
    )

In [ ]:
convert_fit_dir_to_csv("./Source_Data/Fitnessdaten")